# Tool 5 — LLM Agent Orchestrator

An LLM (Claude via API) orchestrates Tools 1–4 in whatever order it judges useful,
then produces a structured diagnostic impression.

**Requires:** `oct_best.pth` and `retrieval_encoder.pth` checkpoints.

**Agent loop:**
```
Image
  → Tool 1: coarse classify         (OCTNet → label + confidence)
  → Tool 2: localize                (GradCAM++ → heatmap + ROI description)
  → Tool 3: zoom-and-reanalyze      (crop ROI → second prediction)
  → Tool 4: retrieve knowledge      (embed query → top-k reference snippets)
  → LLM synthesizes all results
  → Structured output: {finding, confidence, localization, justification}
```


## 0. Setup
Paste imports, config, and model definitions from your main notebook.
Then load both checkpoints.

In [ ]:
import os, random, platform
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score
from sklearn.preprocessing import label_binarize
from tqdm import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IS_WIN = platform.system() == 'Windows'
print(f'Device: {DEVICE} | Platform: {platform.system()}')
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {total_vram:.1f} GB')

DATA_ROOT = Path('OCT2017')
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR  = DATA_ROOT / 'test'

IMG_SIZE        = 224
BATCH_SIZE      = 8
NUM_CLASSES     = 4
EPOCHS          = 60
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.05
WARMUP_EPOCHS   = 5
GRAD_CLIP       = 1.0
CLASS_NAMES     = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CKPT_PATH       = 'oct_best.pth'

NUM_WORKERS = 0 if IS_WIN else 4
PIN_MEMORY  = not IS_WIN and DEVICE.type == 'cuda'
PERSISTENT  = NUM_WORKERS > 0
PREFETCH    = 2 if NUM_WORKERS > 0 else None
print(f'Batch: {BATCH_SIZE} | Workers: {NUM_WORKERS} | pin_memory: {PIN_MEMORY}')


# ── OCTNet architecture ───────────────────────────────────────────────────────
class DSConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, k, padding=p, groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class AnisoBranch(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.h = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, (1,7), padding=(0,3), groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False))
        self.v = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, (7,1), padding=(3,0), groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False))
        self.bn  = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.h(x) + self.v(x)))


class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 4)
        self.ca_mlp  = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(inplace=True), nn.Linear(mid, ch, bias=False))
        self.sa_conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        avg = x.mean([2,3]); mx = x.amax([2,3])
        ca  = torch.sigmoid(self.ca_mlp(avg) + self.ca_mlp(mx))
        x   = x * ca.unsqueeze(-1).unsqueeze(-1)
        sa  = torch.sigmoid(self.sa_conv(torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], dim=1)))
        return x * sa


class MSBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        bc = out_ch // 3
        ex = out_ch - 3 * bc
        self.b3 = DSConv(in_ch, bc + ex, k=3, p=1)
        self.b5 = DSConv(in_ch, bc,      k=5, p=2)
        self.ba = AnisoBranch(in_ch, bc)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.cbam = CBAM(out_ch)
        self.res  = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)) \
                    if in_ch != out_ch else nn.Identity()
        self.act  = nn.SiLU(inplace=True)
    def forward(self, x):
        out = self.cbam(self.bn(torch.cat([self.b3(x), self.b5(x), self.ba(x)], dim=1)))
        return self.act(out + self.res(x))


class OCTNet(nn.Module):
    def __init__(self, num_classes=4, drop=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),           nn.BatchNorm2d(64), nn.SiLU(inplace=True),
        )
        self.s1 = nn.Sequential(MSBlock(64,  96),  nn.MaxPool2d(2), nn.Dropout2d(0.10))
        self.s2 = nn.Sequential(MSBlock(96,  192), MSBlock(192,192), nn.MaxPool2d(2), nn.Dropout2d(0.15))
        self.s3 = nn.Sequential(MSBlock(192, 384), MSBlock(384,384), nn.MaxPool2d(2), nn.Dropout2d(0.20))
        self.s4 = nn.Sequential(MSBlock(384, 512), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(512, 256), nn.SiLU(inplace=True),
            nn.Dropout(drop), nn.Linear(256, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear) and m.weight is not None:
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.s4(self.s3(self.s2(self.s1(self.stem(x))))))

    @property
    def cam_layer(self):
        return self.s4[0].b3.net[0]


class GradCAMpp:
    def __init__(self, model, layer):
        self.model = model; self.grads = None; self.acts = None
        layer.register_forward_hook(lambda m,i,o: setattr(self,'acts',o.detach()))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'grads',go[0].detach()))

    def __call__(self, img_t, cls=None):
        self.model.eval()
        x = img_t.unsqueeze(0).to(DEVICE).requires_grad_(True)
        logits = self.model(x)
        cls = cls if cls is not None else logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, cls].backward()
        g, a = self.grads[0], self.acts[0]
        g2, g3 = g**2, g**3
        alpha = g2 / (2*g2 + (a*g3).sum([1,2], keepdim=True) + 1e-7)
        w = (alpha * F.relu(g)).sum([1,2])
        cam = F.relu((w[:,None,None] * a).sum(0))
        cam = (cam - cam.min()) / (cam.max() + 1e-7)
        conf = torch.softmax(logits, 1)[0].detach().cpu().numpy()
        return cam.cpu().numpy(), cls, conf


val_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])


# ── Tool 4: tokenizer + encoder + corpus ─────────────────────────────────────
class WordTokenizer:
    PAD, UNK = '<PAD>', '<UNK>'

    def __init__(self, texts: list[str], max_len: int = 64):
        self.max_len = max_len
        words = [w.lower().strip('.,();:"\'-') for t in texts for w in t.split()]
        vocab = [self.PAD, self.UNK] + sorted(set(words))
        self.w2i = {w: i for i, w in enumerate(vocab)}
        self.vocab_size = len(vocab)
        print(f'Vocab size: {self.vocab_size}')

    def encode(self, text: str) -> torch.Tensor:
        tokens = [w.lower().strip('.,();:"\'-') for w in text.split()]
        ids = [self.w2i.get(t, 1) for t in tokens]
        ids = ids[:self.max_len] + [0] * max(0, self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)


class SmallEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=4,
                 dim_ff=256, embed_dim=64, max_len=64, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.proj = nn.Linear(d_model, embed_dim, bias=False)
        nn.init.normal_(self.tok_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, input_ids):
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, -1)
        padding_mask = (input_ids == 0)
        x = self.tok_emb(input_ids) + self.pos_emb(positions)
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        mask = (~padding_mask).float().unsqueeze(-1)
        x = (x * mask).sum(1) / mask.sum(1).clamp(min=1)
        return F.normalize(self.proj(x), dim=-1)


# Load expanded corpus from JSON 
import json

with open('oct_corpus_xl.json') as f:
    corpus_data = json.load(f)

CORPUS = [(item['label'], item['text']) for item in corpus_data]
print(f'Loaded {len(CORPUS)} corpus entries')

from collections import Counter
counts = Counter(label for label, _ in CORPUS)
for label, n in counts.items():
    print(f'  {label}: {n}')

# ── Build tokenizer BEFORE encoder ───────────────────────────────────────────
all_texts = [t for _, t in CORPUS] + [l for l, _ in CORPUS]
tokenizer = WordTokenizer(all_texts, max_len=64)


@torch.no_grad()
def embed_texts(texts: list[str]) -> torch.Tensor:
    ids = torch.stack([tokenizer.encode(t) for t in texts]).to(DEVICE)
    return encoder(ids).cpu()


@torch.no_grad()
def retrieve(query: str, top_k: int = 3, label_filter=None) -> list[dict]:
    q_emb = embed_texts([query])
    scores = (corpus_embs @ q_emb.T).squeeze(1)
    if label_filter:
        mask = torch.tensor([l == label_filter for l in corpus_labels])
        scores = scores.masked_fill(~mask, -1.0)
    top_idx = scores.argsort(descending=True)[:top_k]
    return [{'rank': rank+1, 'label': corpus_labels[i],
             'text': corpus_texts[i], 'score': float(scores[i])}
            for rank, i in enumerate(top_idx.tolist())]


# ── Tool 3: zoom helpers ──────────────────────────────────────────────────────
def get_cam_bbox(cam: np.ndarray, threshold: float = 0.4):
    binary = (cam >= threshold).astype(np.uint8)
    rows = np.any(binary, axis=1)
    cols = np.any(binary, axis=0)
    if not rows.any() or not cols.any():
        h, w = cam.shape
        return w//4, h//4, 3*w//4, 3*h//4
    y0, y1 = np.where(rows)[0][[0, -1]]
    x0, x1 = np.where(cols)[0][[0, -1]]
    return int(x0), int(y0), int(x1), int(y1)


def zoom_and_reanalyze(pil_image, model, cam_fn, class_names,
                       img_size=224, cam_threshold=0.4, padding_frac=0.10):
    infer_tf = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    img_t = infer_tf(pil_image)
    cam, orig_cls, orig_conf = cam_fn(img_t)

    cam_h, cam_w = cam.shape
    cx0, cy0, cx1, cy1 = get_cam_bbox(cam, cam_threshold)
    x0 = max(0,        int(cx0 * img_size/cam_w) - int(img_size*padding_frac))
    y0 = max(0,        int(cy0 * img_size/cam_h) - int(img_size*padding_frac))
    x1 = min(img_size, int(cx1 * img_size/cam_w) + int(img_size*padding_frac))
    y1 = min(img_size, int(cy1 * img_size/cam_h) + int(img_size*padding_frac))

    crop_t = img_t[:, y0:y1, x0:x1].unsqueeze(0)
    crop_t = F.interpolate(crop_t, size=(img_size, img_size), mode='bilinear', align_corners=False)
    crop_t = crop_t.squeeze(0)
    _, crop_cls, crop_conf = cam_fn(crop_t)

    return {
        'original_pred':  class_names[orig_cls],
        'original_conf':  float(orig_conf[orig_cls]),
        'original_probs': {c: float(p) for c, p in zip(class_names, orig_conf)},
        'cam': cam, 'bbox': (x0, y0, x1, y1),
        'crop_pred':  class_names[crop_cls],
        'crop_conf':  float(crop_conf[crop_cls]),
        'crop_probs': {c: float(p) for c, p in zip(class_names, crop_conf)},
        'agreement':  orig_cls == crop_cls,
    }


# ── Load models ───────────────────────────────────────────────────────────────
eval_model = OCTNet(NUM_CLASSES).to(DEVICE)
eval_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
eval_model.eval()
cam_fn = GradCAMpp(eval_model, eval_model.cam_layer)

CKPT_EMB = 'retrieval_encoder.pth'
encoder = SmallEncoder(vocab_size=tokenizer.vocab_size).to(DEVICE)
encoder.load_state_dict(torch.load(CKPT_EMB, map_location=DEVICE))
encoder.eval()

corpus_texts  = [text  for _, text  in CORPUS]
corpus_labels = [label for label, _ in CORPUS]
corpus_embs   = embed_texts(corpus_texts)
print('All models loaded.')

## 1. Tool Definitions
Wrap each capability as a clean function the agent can call by name.

In [ ]:
import json
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms


def tool_coarse_classify(image_path: str) -> dict:
    """Tool 1: Run OCTNet on the full image."""
    infer_tf = transforms.Compose([
        transforms.Grayscale(3),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    pil = Image.open(image_path)
    img_t = infer_tf(pil)
    cam, cls_idx, probs = cam_fn(img_t)
    return {
        'predicted_class': CLASS_NAMES[cls_idx],
        'confidence':      round(float(probs[cls_idx]), 4),
        'all_probs':       {c: round(float(p), 4) for c, p in zip(CLASS_NAMES, probs)},
    }


def tool_localize(image_path: str) -> dict:
    """Tool 2: Run GradCAM++ and describe the activated region."""
    infer_tf = transforms.Compose([
        transforms.Grayscale(3),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    pil = Image.open(image_path)
    img_t = infer_tf(pil)
    cam, cls_idx, probs = cam_fn(img_t)

    h, w = cam.shape
    peak_y, peak_x = divmod(int(cam.argmax()), w)
    frac_x, frac_y = peak_x / w, peak_y / h
    horiz = 'nasal' if frac_x < 0.4 else ('temporal' if frac_x > 0.6 else 'central')
    vert  = 'superior' if frac_y < 0.4 else ('inferior' if frac_y > 0.6 else 'mid')
    region = f'{vert}-{horiz}' if horiz != 'central' or vert != 'mid' else 'central foveal'
    activation_spread = float((cam > 0.4).sum()) / cam.size

    return {
        'predicted_class':    CLASS_NAMES[cls_idx],
        'peak_activation_at': region,
        'activation_spread':  round(activation_spread, 3),
        'spread_description': 'focal' if activation_spread < 0.15 else ('moderate' if activation_spread < 0.35 else 'diffuse'),
    }


def tool_zoom_reanalyze(image_path: str) -> dict:
    """Tool 3: Crop the GradCAM-activated region and re-classify."""
    pil = Image.open(image_path)
    result = zoom_and_reanalyze(pil, eval_model, cam_fn, CLASS_NAMES)
    return {
        'full_image_pred': result['original_pred'],
        'full_image_conf': round(result['original_conf'], 4),
        'crop_pred':       result['crop_pred'],
        'crop_conf':       round(result['crop_conf'], 4),
        'agreement':       result['agreement'],
        'bbox':            result['bbox'],
    }


def tool_retrieve_knowledge(query: str, top_k: int = 3) -> dict:
    """Tool 4: Retrieve reference radiology text relevant to a query."""
    results = retrieve(query, top_k=top_k)
    return {
        'query':   query,
        'results': [{'rank': r['rank'], 'label': r['label'],
                     'text': r['text'], 'score': round(r['score'], 4)} for r in results],
    }


TOOL_REGISTRY = {
    'coarse_classify':    tool_coarse_classify,
    'localize':           tool_localize,
    'zoom_reanalyze':     tool_zoom_reanalyze,
    'retrieve_knowledge': tool_retrieve_knowledge,
}

print('Tool registry ready:', list(TOOL_REGISTRY.keys()))


## 2. System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are a diagnostic AI assistant for OCT retinal imaging.
You have access to four tools:

1. coarse_classify(image_path) -> {predicted_class, confidence, all_probs}
   Run a CNN classifier on the full image. Classes: CNV, DME, DRUSEN, NORMAL.

2. localize(image_path) -> {predicted_class, peak_activation_at, spread_description}
   Run GradCAM++ to identify which region drove the prediction.

3. zoom_reanalyze(image_path) -> {full_image_pred, crop_pred, agreement, crop_conf}
   Crop the activated region and re-run the classifier on it.
   Use when confidence is below 85% or finding is ambiguous.

4. retrieve_knowledge(query, top_k=3) -> {results: [{label, text, score}]}
   Retrieve reference radiology text. E.g. query: 'CNV finding', 'subretinal fluid'.

INSTRUCTIONS:
- Always start with coarse_classify and localize.
- Call zoom_reanalyze if confidence < 85% or finding is ambiguous.
- Call retrieve_knowledge for the predicted class and runner-up if probs are within 15%.
- You may call tools multiple times in any order.
- Produce a final structured JSON with these exact keys:
  {
    "finding": "<CNV|DME|DRUSEN|NORMAL>",
    "confidence": <0.0-1.0>,
    "localization": "<where in the image>",
    "supporting_evidence": ["<clinical features that match>"],
    "justification": "<2-3 sentences referencing each tool>",
    "uncertainty_flags": ["<low confidence, disagreement, etc>"]
  }

Call tools with:
TOOL: <tool_name>
ARGS: <JSON args>

When done:
FINAL: <JSON output>
"""

print('System prompt defined.')

## 3. Agent Loop

In [ ]:
import os
import re
import json
import time
from cerebras.cloud.sdk import Cerebras

API_KEY = os.getenv("CEREBRAS_API_KEY", "")
client = Cerebras(api_key=API_KEY)

FAST_MODE = os.getenv("FAST_MODE", "true").lower() in {"1", "true", "yes", "on"}
MODEL_NAME = os.getenv("CEREBRAS_MODEL", "gpt-oss-120b")
MAX_TURNS = int(os.getenv("MAX_TURNS", "4" if FAST_MODE else "8"))
MAX_COMPLETION_TOKENS = int(os.getenv("MAX_COMPLETION_TOKENS", "300" if FAST_MODE else "700"))
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0" if FAST_MODE else "0.2"))
REASONING_EFFORT = os.getenv("REASONING_EFFORT", "low" if FAST_MODE else "medium")
print(f"Agent mode: {'fast' if FAST_MODE else 'full'} | model={MODEL_NAME} | max_turns={MAX_TURNS}")


def parse_tool_call(text: str):
    tool_m = re.search(r'TOOL:\s*(\w+)', text)
    args_m = re.search(r'ARGS:\s*(\{.*?\})', text, re.DOTALL)
    if not tool_m:
        return None
    tool_name = tool_m.group(1).strip()
    args = json.loads(args_m.group(1)) if args_m else {}
    return tool_name, args


def parse_final(text: str):
    m = re.search(r'FINAL:\s*(\{.*\})', text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None


def build_fast_fallback(image_path: str) -> dict:
    coarse = tool_coarse_classify(image_path)
    local = tool_localize(image_path)
    probs = coarse.get("all_probs", {})
    ranked = sorted(probs.items(), key=lambda kv: kv[1], reverse=True) if probs else []
    top_conf = ranked[0][1] if ranked else 0.0
    runner_conf = ranked[1][1] if len(ranked) > 1 else 0.0
    should_skip_llm = coarse["confidence"] >= 0.90 and (top_conf - runner_conf) >= 0.15

    flags = []
    if coarse["confidence"] < 0.85:
        flags.append("low confidence")
    if probs and (top_conf - runner_conf) < 0.15:
        flags.append("close runner-up")

    return {
        "finding": coarse["predicted_class"],
        "confidence": coarse["confidence"],
        "localization": local["peak_activation_at"],
        "supporting_evidence": [f"{coarse['predicted_class']} with {local['spread_description']} activation"],
        "justification": (
            f"The CNN classifier predicted {coarse['predicted_class']} with confidence "
            f"{coarse['confidence']:.2f}. GradCAM++ localized the dominant activation to "
            f"{local['peak_activation_at']} with {local['spread_description']} spread."
        ),
        "uncertainty_flags": flags,
        "_tool_calls": [
            {"tool": "coarse_classify", "args": {"image_path": image_path}, "result": coarse},
            {"tool": "localize", "args": {"image_path": image_path}, "result": local},
        ],
        "_fast_fallback": should_skip_llm,
    }


def run_agent(image_path: str, verbose: bool = True) -> dict:
    if FAST_MODE:
        fast_result = build_fast_fallback(image_path)
        if fast_result["_fast_fallback"]:
            if verbose:
                print("Fast path used; skipping LLM for this high-confidence case.")
            return fast_result

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze this OCT image and provide a structured diagnostic impression.\nImage path: {image_path}"}
    ]

    tool_calls_log = []

    for turn in range(MAX_TURNS):
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    max_completion_tokens=MAX_COMPLETION_TOKENS,
                    temperature=TEMPERATURE,
                    top_p=1,
                    stream=False,
                    reasoning_effort=REASONING_EFFORT,
                )
                break
            except Exception as e:
                if '429' in str(e) or 'queue_exceeded' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                    wait = 8 * (attempt + 1)
                    print(f'Rate limited — waiting {wait}s (attempt {attempt+1}/3)...')
                    time.sleep(wait)
                else:
                    raise
        else:
            return {'error': 'Rate limit retries exhausted', '_tool_calls': tool_calls_log}

        assistant_text = response.choices[0].message.content
        messages.append({"role": "assistant", "content": assistant_text})

        final = parse_final(assistant_text)
        if final:
            final['_tool_calls'] = tool_calls_log
            if verbose:
                print(f'\n=== FINAL OUTPUT (after {turn+1} turns) ===')
                print(json.dumps(final, indent=2))
            return final

        parsed = parse_tool_call(assistant_text)
        if not parsed:
            break

        tool_name, args = parsed
        if tool_name not in TOOL_REGISTRY:
            tool_result = {'error': f'Unknown tool: {tool_name}'}
        else:
            try:
                tool_result = TOOL_REGISTRY[tool_name](**args)
            except Exception as e:
                tool_result = {'error': str(e)}

        tool_calls_log.append({'tool': tool_name, 'args': args, 'result': tool_result})

        if verbose:
            print(f'Turn {turn+1} | {tool_name} -> {json.dumps(tool_result)[:120]}...')

        messages.append({
            "role": "user",
            "content": f"TOOL_RESULT for {tool_name}:\n{json.dumps(tool_result, indent=2)}"
        })

    return {'error': 'Agent did not produce final output', '_tool_calls': tool_calls_log}

print('run_agent() with fast fallback and reduced traffic defined.')

## 4. Run on a single example

In [ ]:
test_ds = datasets.ImageFolder(TEST_DIR, transform=val_tf)
print(f'Test set: {len(test_ds)} images')
print(f'Classes: {test_ds.classes}')

In [ ]:
sample_path, sample_label = test_ds.samples[0]
true_class = CLASS_NAMES[sample_label]
print(f'Image: {sample_path}')
print(f'True label: {true_class}\n')

result = run_agent(sample_path, verbose=True)

print(f'\n-> Predicted : {result.get("finding")}')
print(f'-> True      : {true_class}')
print(f'-> Correct   : {result.get("finding") == true_class}')


## 5. Ablation Study
Remove each tool in turn and measure accuracy + F1 drop.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm


def ablation_run(image_path: str, disabled_tools: list) -> str:
    originals = {}
    for t in disabled_tools:
        originals[t] = TOOL_REGISTRY[t]
        TOOL_REGISTRY[t] = lambda **_: {'error': 'tool disabled for ablation'}
    result = run_agent(image_path, verbose=False)
    for t, fn in originals.items():
        TOOL_REGISTRY[t] = fn
    return result.get('finding', 'UNKNOWN')


# Build evaluation subset: 40 images per class
subset = []
per_class = {c: 0 for c in CLASS_NAMES}
for path, label in test_ds.samples:
    cname = CLASS_NAMES[label]
    if per_class[cname] < 40:
        subset.append((path, label))
        per_class[cname] += 1
print(f'Ablation subset: {len(subset)} images')

ABLATION_CONDITIONS = {
    'Full pipeline':            [],
    'No zoom (-Tool3)':         ['zoom_reanalyze'],
    'No retrieval (-Tool4)':    ['retrieve_knowledge'],
    'No localization (-Tool2)': ['localize'],
    'Classifier only (-2,3,4)': ['localize', 'zoom_reanalyze', 'retrieve_knowledge'],
}

ablation_results = {}
for condition, disabled in ABLATION_CONDITIONS.items():
    preds, trues = [], []
    for path, label in tqdm(subset, desc=condition):
        pred = ablation_run(path, disabled)
        preds.append(pred)
        trues.append(CLASS_NAMES[label])
    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average='macro', labels=CLASS_NAMES, zero_division=0)
    ablation_results[condition] = {'accuracy': acc, 'macro_f1': f1}
    print(f'{condition:35s} | Acc {acc*100:.1f}% | F1 {f1*100:.1f}%')


## 6. Ablation Table

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {'Condition': cond,
     'Accuracy (%)': f"{v['accuracy']*100:.1f}",
     'Macro F1 (%)': f"{v['macro_f1']*100:.1f}"}
    for cond, v in ablation_results.items()
])
print(df.to_string(index=False))


## 7. Visualize Agent Trace

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage


def visualize_agent_trace(image_path: str, result: dict):
    tool_calls = result.get('_tool_calls', [])

    infer_tf = transforms.Compose([
        transforms.Grayscale(3), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    pil = PILImage.open(image_path)
    img_t = infer_tf(pil)
    cam, _, _ = cam_fn(img_t)
    img_gray = (img_t.numpy().transpose(1, 2, 0) * 0.5 + 0.5).mean(2)
    cam_up   = np.array(PILImage.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.
    overlay  = (0.55 * np.stack([img_gray]*3, 2) + 0.45 * plt.cm.jet(cam_up)[:,:,:3]).clip(0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        f"Agent Diagnosis: {result.get('finding', '?')}  "
        f"(conf={result.get('confidence', 0)*100:.0f}%)  "
        f"| {len(tool_calls)} tool calls",
        fontsize=13
    )

    axes[0].imshow(img_gray, cmap='gray'); axes[0].set_title('Input OCT'); axes[0].axis('off')
    axes[1].imshow(overlay);               axes[1].set_title('GradCAM++ overlay'); axes[1].axis('off')

    axes[2].axis('off')
    justification = result.get('justification') or ''
    evidence = result.get('supporting_evidence') or []
    flags = result.get('uncertainty_flags') or []
    summary = (
        [f"Finding    : {result.get('finding', '?')}",
         f"Confidence : {result.get('confidence', 0)*100:.0f}%",
         f"Location   : {result.get('localization', 'N/A')}",
         "", "Justification:"]
        + [f"  {s.strip()}." for s in justification.split('.') if s.strip()]
        + ["", "Supporting evidence:"]
        + [f"  * {e}" for e in evidence[:3]]
        + [""]
        + ([f"!! {f}" for f in flags] or ['OK No uncertainty flags'])
    )
    axes[2].text(0.02, 0.98, '\n'.join(summary), transform=axes[2].transAxes,
                 fontsize=8, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.savefig('tool5_agent_trace.png', dpi=150, bbox_inches='tight')
    plt.show()


visualize_agent_trace(sample_path, result)


In [ ]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np
from tqdm import tqdm

# Your model's classes — must match CKPT_PATH training
MODEL_CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']


def load_shared_dataset(folder: str, model_classes: list, transform) -> datasets.ImageFolder:
    """
    Load an ImageFolder dataset but keep ONLY classes the model knows.
    Remaps class indices so they match MODEL_CLASSES order.
    """
    full_ds = datasets.ImageFolder(folder, transform=transform)
    available = set(full_ds.classes)
    shared    = [c for c in model_classes if c in available]
    missing   = [c for c in model_classes if c not in available]

    print(f'Folder       : {folder}')
    print(f'All classes  : {full_ds.classes}')
    print(f'Shared       : {shared}')
    print(f'Not in folder: {missing}')

    # Keep only samples whose class is in shared
    shared_set = set(shared)
    indices = [i for i, (_, label_idx) in enumerate(full_ds.samples)
               if full_ds.classes[label_idx] in shared_set]

    # Build a filtered subset
    subset = Subset(full_ds, indices)

    # Store metadata for easy access
    subset.shared_classes    = shared
    subset.full_ds           = full_ds
    subset.model_classes     = model_classes

    # Build a mapping: original label index -> model class index
    subset.label_remap = {
        full_ds.class_to_idx[c]: model_classes.index(c)
        for c in shared
    }

    print(f'Kept samples : {len(indices)}')
    return subset


def get_sample_list(subset) -> list:
    """Return list of (path, remapped_label) for a filtered subset."""
    samples = []
    for idx in subset.indices:
        path, orig_label = subset.full_ds.samples[idx]
        remapped = subset.label_remap[orig_label]
        samples.append((path, remapped))
    return samples


def normalize_prediction_name(pred, fallback=None):
    """Normalize agent outputs to one of MODEL_CLASSES or return a safe fallback."""
    if pred is None:
        return fallback

    if not isinstance(pred, str):
        pred = str(pred)

    text = pred.strip()
    if not text:
        return fallback

    key = ''.join(ch for ch in text.upper() if ch.isalnum())
    aliases = {
        'CNV': 'CNV',
        'DME': 'DME',
        'DRUSEN': 'DRUSEN',
        'NORMAL': 'NORMAL',
        'NORM': 'NORMAL',
        'UNKNOWN': fallback,
        'ERROR': fallback,
        'NONE': fallback,
    }
    if key in aliases:
        return aliases[key]

    for cls in MODEL_CLASSES:
        if cls.upper() == key:
            return cls

    if key.startswith('CN'):
        return 'CNV'
    if key.startswith('DM'):
        return 'DME'
    if key.startswith('DR'):
        return 'DRUSEN'
    if key.startswith('NOR'):
        return 'NORMAL'
    return fallback


def prediction_to_index(pred, fallback=-1):
    """Convert a prediction name to an index, using a safe fallback on invalid values."""
    name = normalize_prediction_name(pred, fallback=None)
    if name is None:
        return fallback
    try:
        return MODEL_CLASSES.index(name)
    except ValueError:
        return fallback


print('Helper functions defined.')

In [ ]:
OCTC8_TEST = 'octc8/test'   # ← change path if needed

octc8_subset = load_shared_dataset(OCTC8_TEST, MODEL_CLASSES, val_tf)
octc8_samples = get_sample_list(octc8_subset)
print(f'\noctc8 test samples (shared classes): {len(octc8_samples)}')
OCTID_ROOT = 'octid'   # ← change path if needed

octid_subset = load_shared_dataset(OCTID_ROOT, MODEL_CLASSES, val_tf)
octid_samples = get_sample_list(octid_subset)
print(f'\noctid samples (shared classes): {len(octid_samples)}')



In [ ]:
from PIL import Image as PILImage


def evaluate_classifier_only(samples: list, dataset_name: str) -> dict:
    """Run OCTNet directly (no agent) and report accuracy + F1."""
    preds, trues = [], []
    for path, true_label in tqdm(samples, desc=f'Classifier eval [{dataset_name}]'):
        result = tool_coarse_classify(path)
        pred_label = prediction_to_index(result.get('predicted_class', 'UNKNOWN'))
        if pred_label < 0:
            pred_label = prediction_to_index(result.get('finding', 'UNKNOWN'))
        preds.append(pred_label)
        trues.append(true_label)

    labels = sorted(set(trues) | set(preds))
    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average='macro',
                   labels=labels, zero_division=0)
    print(f'\n=== {dataset_name} — Classifier Only ===')
    print(f'Accuracy : {acc*100:.2f}%')
    print(f'Macro F1 : {f1*100:.2f}%')
    target_names = [MODEL_CLASSES[i] for i in labels if 0 <= i < len(MODEL_CLASSES)]
    print(classification_report(trues, preds, labels=labels, target_names=target_names,
                                 zero_division=0, digits=4))
    return {'accuracy': acc, 'macro_f1': f1, 'preds': preds, 'trues': trues}


octc8_baseline = evaluate_classifier_only(octc8_samples, 'octc8')

octid_baseline = evaluate_classifier_only(octid_samples, 'octid')

In [ ]:
from collections import defaultdict


def ablation_run(image_path: str, disabled_tools: list) -> str:
    originals = {}
    for t in disabled_tools:
        originals[t] = TOOL_REGISTRY[t]
        TOOL_REGISTRY[t] = lambda **_: {'error': 'tool disabled for ablation'}
    try:
        result = run_agent(image_path, verbose=False)
        pred_name = normalize_prediction_name(
            result.get('finding') or result.get('predicted_class') or result.get('predicted') or 'UNKNOWN',
            fallback=None,
        )
        if pred_name not in MODEL_CLASSES:
            try:
                pred_name = tool_coarse_classify(image_path)['predicted_class']
            except Exception:
                pred_name = MODEL_CLASSES[0]
    except Exception:
        pred_name = MODEL_CLASSES[0]
    finally:
        for t, fn in originals.items():
            TOOL_REGISTRY[t] = fn
    return pred_name


# Build balanced subset — up to 40 images per shared class
per_class = defaultdict(list)
for path, label in octc8_samples:
    if len(per_class[label]) < 40:
        per_class[label].append((path, label))

ablation_subset = [item for items in per_class.values() for item in items]
print(f'Ablation subset: {len(ablation_subset)} images')
for label_idx, items in per_class.items():
    print(f'  {MODEL_CLASSES[label_idx]}: {len(items)} images')


In [ ]:
import time

ABLATION_CONDITIONS = {
    'Full pipeline':            [],
    'No zoom (-Tool3)':         ['zoom_reanalyze'],
    'No retrieval (-Tool4)':    ['retrieve_knowledge'],
    'No localization (-Tool2)': ['localize'],
    'Classifier only (-2,3,4)': ['localize', 'zoom_reanalyze', 'retrieve_knowledge'],
}

ablation_results_octc8 = {}

for condition, disabled in ABLATION_CONDITIONS.items():
    preds, trues = [], []
    for path, label in tqdm(ablation_subset, desc=condition):
        pred_name = ablation_run(path, disabled)
        pred_idx  = MODEL_CLASSES.index(pred_name) if pred_name in MODEL_CLASSES else -1
        preds.append(pred_idx)
        trues.append(label)
        time.sleep(0.5)   # small delay to avoid rate limits

    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average='macro',
                   labels=list(range(len(MODEL_CLASSES))), zero_division=0)
    ablation_results_octc8[condition] = {'accuracy': acc, 'macro_f1': f1}
    print(f'{condition:35s} | Acc {acc*100:.1f}% | F1 {f1*100:.1f}%')


In [ ]:
print('Finding uncertain cases (confidence < 85%)...')
uncertain = []
for path, label in tqdm(octc8_samples, desc='Scanning confidence'):
    result = tool_coarse_classify(path)
    if result['confidence'] < 0.85:
        uncertain.append((path, label, result['confidence']))

print(f'Uncertain cases: {len(uncertain)} / {len(octc8_samples)} ({len(uncertain)/len(octc8_samples)*100:.1f}%)')
for path, label, conf in uncertain[:10]:
    print(f'  {MODEL_CLASSES[label]} | conf={conf:.3f} | {Path(path).name}')


In [ ]:
if len(uncertain) == 0:
    print('No uncertain cases found — classifier is highly confident on octc8.')
    print('This is a strong generalization result but limits ablation signal.')
else:
    uncertain_samples = [(p, l) for p, l, _ in uncertain]
    ablation_results_uncertain = {}

    for condition, disabled in ABLATION_CONDITIONS.items():
        preds, trues = [], []
        for path, label in tqdm(uncertain_samples, desc=f'[uncertain] {condition}'):
            pred_name = ablation_run(path, disabled)
            pred_idx  = MODEL_CLASSES.index(pred_name) if pred_name in MODEL_CLASSES else -1
            preds.append(pred_idx)
            trues.append(label)
            time.sleep(0.5)

        acc = accuracy_score(trues, preds)
        f1  = f1_score(trues, preds, average='macro',
                       labels=list(range(len(MODEL_CLASSES))), zero_division=0)
        ablation_results_uncertain[condition] = {'accuracy': acc, 'macro_f1': f1}
        print(f'{condition:35s} | Acc {acc*100:.1f}% | F1 {f1*100:.1f}%')


In [ ]:
import pandas as pd

print('\n=== CLASSIFIER BASELINE (no agent) ===')
baseline_df = pd.DataFrame([
    {'Dataset': 'OCT2017 (original)', 'Accuracy (%)': '99.4', 'Macro F1 (%)': '99.7'},
    {'Dataset': 'octc8 shared classes', 
     'Accuracy (%)': f"{octc8_baseline['accuracy']*100:.1f}",
     'Macro F1 (%)': f"{octc8_baseline['macro_f1']*100:.1f}"},
    {'Dataset': 'octid shared classes',
     'Accuracy (%)': f"{octid_baseline['accuracy']*100:.1f}",
     'Macro F1 (%)': f"{octid_baseline['macro_f1']*100:.1f}"},
])
print(baseline_df.to_string(index=False))

print('\n=== ABLATION — octc8 shared classes ===')
ablation_df = pd.DataFrame([
    {'Condition': cond,
     'Accuracy (%)': f"{v['accuracy']*100:.1f}",
     'Macro F1 (%)': f"{v['macro_f1']*100:.1f}"}
    for cond, v in ablation_results_octc8.items()
])
print(ablation_df.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


def plot_cm(trues, preds, class_names, title, filename):
    cm = confusion_matrix(trues, preds, labels=list(range(len(class_names))))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()


plot_cm(octc8_baseline['trues'], octc8_baseline['preds'],
        MODEL_CLASSES, 'OCTNet on octc8 (shared classes)', 'cm_octc8.png')

plot_cm(octid_baseline['trues'], octid_baseline['preds'],
        MODEL_CLASSES, 'OCTNet on octid (shared classes)', 'cm_octid.png')
